## Root Cause Analysis - Exp4

* Goals
  * Check whether can provide improvement suggestions at individual cases level


In [35]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import psycopg2
from utils_root_cause import upload_to_google_drive

import warnings
warnings.filterwarnings('ignore')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# Environment variable holding the database URL
DB_ENV = "DATABASE_URL_PUBLIC_RAG_DR"
db_url = os.getenv(DB_ENV)
if not db_url:
    raise RuntimeError(f"Environment variable {DB_ENV} is not set")

# Connect and load the table into a pandas DataFrame
conn = psycopg2.connect(db_url)
try:
    df_existing_rca_output = pd.read_sql_query("SELECT * FROM existing_rca_output", conn)
finally:
    try:
        conn.close()
    except Exception:
        pass

# Display the first rows
print(df_existing_rca_output.shape)
df_existing_rca_output.head()

C:\Users\wuhan\AppData\Local\Temp\ipykernel_17640\1673439458.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_existing_rca_output = pd.read_sql_query("SELECT * FROM existing_rca_output", conn)


(4, 5)


,id,config_hash,rca_records,agg_review,created_at
0,1,93f67870e1be18f0328798b37faf07df,[{'query': 'How to deposit a cheque issued to ...,{'root_cause_analysis': 'Limited retrieval (To...,2026-04-13 04:09:33.082586
1,2,fac62bbce5555de5ca44b01442baea3f,[{'query': 'How to deposit a cheque issued to ...,{'root_cause_analysis': 'Retrieval is high qua...,2026-04-13 04:28:22.134432
2,3,cda0e838de11a82a1802e38b8b0d9ab9,[{'query': 'How to deposit a cheque issued to ...,"{'root_cause_analysis': 'Low semantic weight, ...",2026-04-13 17:09:46.469203
3,4,9c99877c5c2cde6ac3a5c05ea242b577,[{'query': 'How to deposit a cheque issued to ...,{'root_cause_analysis': 'Answer quality suffer...,2026-04-13 18:51:48.106385


In [3]:
# Convert any list-valued columns so each item becomes its own row
df = df_existing_rca_output.copy()
# Find columns that contain lists in at least one row
list_cols = [c for c in df.columns if df[c].apply(lambda x: isinstance(x, list)).any()]
if not list_cols:
    print('No list-valued columns found; no conversion needed')
else:
    for col in list_cols:
        # Explode the list so each element becomes a separate row
        df = df.explode(col).reset_index(drop=True)
        # If the exploded elements are dict-like, expand their keys into columns
        if df[col].dropna().apply(lambda x: isinstance(x, dict)).any():
            expanded = pd.json_normalize(df[col]).add_prefix(f'{col}.')
            df = pd.concat([df.drop(columns=[col]).reset_index(drop=True), expanded.reset_index(drop=True)], axis=1)
    df_records_rows = df
    print('Converted list-valued columns to rows. Result shape:', df_records_rows.shape)
    display(df_records_rows.head())

Converted list-valued columns to rows. Result shape: (120, 19)


,id,config_hash,agg_review,created_at,rca_records.query,rca_records.context,rca_records.ai_answer,rca_records.aq_reasoning,rca_records.rq_reasoning,rca_records.same_context,rca_records.needs_re_eval,rca_records.query_quality,rca_records.referenced_answer,rca_records.retrieved_content,rca_records.root_cause_analysis,rca_records.answer_quality_score,rca_records.retrieval_quality_score,rca_records.new_answer_quality_score,rca_records.new_retrieval_quality_score
0,1,93f67870e1be18f0328798b37faf07df,{'root_cause_analysis': 'Limited retrieval (To...,2026-04-13 04:09:33.082586,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,Have the associate endorse the back of the che...,The AI’s answer is fully relevant to the user’...,The retrieved content is identical to the cont...,True,0,clear,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...,[],3,3,3,3
1,1,93f67870e1be18f0328798b37faf07df,{'root_cause_analysis': 'Limited retrieval (To...,2026-04-13 04:09:33.082586,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,Yes. You can purchase a USPS money order and l...,The AI’s answer fully addresses the user’s que...,The retrieved content matches the context exac...,True,0,clear,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,[],3,3,3,3
2,1,93f67870e1be18f0328798b37faf07df,{'root_cause_analysis': 'Limited retrieval (To...,2026-04-13 04:09:33.082586,1 EIN doing business under multiple business n...,You're confusing a lot of things here. Company...,You can operate several “doing‑business‑as” (D...,The AI’s answer directly addresses the user’s ...,The retrieved content is identical to the cont...,True,0,ambiguous,You're confusing a lot of things here. Company...,You're confusing a lot of things here. Company...,[{'Please review query quality': ['Can one EIN...,3,3,3,3
3,1,93f67870e1be18f0328798b37faf07df,{'root_cause_analysis': 'Limited retrieval (To...,2026-04-13 04:09:33.082586,Applying for and receiving business credit,Set up a meeting with the bank that handles yo...,"To obtain business credit, follow these practi...",The AI’s answer is fully relevant to the user’...,The retrieved content reproduces the entire co...,True,0,clear,"""I'm afraid the great myth of limited liabilit...",Set up a meeting with the bank that handles yo...,[],3,3,3,3
4,1,93f67870e1be18f0328798b37faf07df,{'root_cause_analysis': 'Limited retrieval (To...,2026-04-13 04:09:33.082586,401k Transfer After Business Closure,The time horizon for your 401K/IRA is essentia...,"When a company shuts down, the retirement plan...",The AI’s answer correctly explains the rollove...,The retrieved content mirrors the context exac...,False,0,clear,You should probably consult an attorney. Howev...,The time horizon for your 401K/IRA is essentia...,[{'auto eval score issue': 'score vs reasoning...,1,3,2,3


In [5]:
folder_id = os.getenv('GOOGLE_DRIVE_FOLDER_ID')
upload_to_google_drive(df_records_rows, folder_id, 'rca_revisit_v1.xlsx')

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=799250539083-ihi7rtknhm6oq5niabbk1nh6bi506air.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A64941%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.file&state=JC1kpHgRZhK9GLimGo6Rg3XCHC4Gwd&code_challenge=-kz5GMlECWvnE1ulSETSm_gW1PYztMEzHXb7Asx-lRA&code_challenge_method=S256&access_type=offline&prompt=consent
Uploaded file ID: 1PQyr8-ocML3TKCChTisH-BqYh-H6VHQp


## Data Analysis

In [14]:
dist_df = pd.DataFrame(df_records_rows[['config_hash', 
                'rca_records.new_retrieval_quality_score',
                'rca_records.new_answer_quality_score']].value_counts()).reset_index()
dist_df.sort_values(by=['config_hash', 'count'], ascending=[True, False], inplace=True)

dist_df

,config_hash,rca_records.new_retrieval_quality_score,rca_records.new_answer_quality_score,count
1,93f67870e1be18f0328798b37faf07df,3,3,16
5,93f67870e1be18f0328798b37faf07df,3,2,10
15,93f67870e1be18f0328798b37faf07df,3,-1,1
16,93f67870e1be18f0328798b37faf07df,2,2,1
17,93f67870e1be18f0328798b37faf07df,3,1,1
18,93f67870e1be18f0328798b37faf07df,3,0,1
2,9c99877c5c2cde6ac3a5c05ea242b577,3,3,15
4,9c99877c5c2cde6ac3a5c05ea242b577,3,2,11
10,9c99877c5c2cde6ac3a5c05ea242b577,3,1,2
13,9c99877c5c2cde6ac3a5c05ea242b577,3,-1,1


In [20]:
# Sample and inspect data from `existing_rag_output`
import os
import pandas as pd
import psycopg2

DB_ENV = "DATABASE_URL_PUBLIC_RAG_DR"
db_url = os.getenv(DB_ENV)
if not db_url:
    raise RuntimeError(f"Environment variable {DB_ENV} is not set")

conn = psycopg2.connect(db_url)
try:
    disinct_settings = pd.read_sql_query("""SELECT distinct config_hash, created_at, embedding_model,
                                               top_n_retrieval, semantic_weight, answer_gen_llm
                                               FROM existing_rag_output 
                                            order by semantic_weight""", conn)
finally:
    try:
        conn.close()
    except Exception:
        pass

print('existing_rag_output shape:', disinct_settings.shape)
print('Columns:', list(disinct_settings.columns))
display(disinct_settings)

existing_rag_output shape: (6, 6)
Columns: ['config_hash', 'created_at', 'embedding_model', 'top_n_retrieval', 'semantic_weight', 'answer_gen_llm']


,config_hash,created_at,embedding_model,top_n_retrieval,semantic_weight,answer_gen_llm
0,cda0e838de11a82a1802e38b8b0d9ab9,2026-04-13 01:10:37.311686,text-embedding-3-small,1,0.1,llama-3.1-8b-instant
1,724dfb1dd56c14fbb33aaae23adf4627,2026-04-13 01:14:53.818664,text-embedding-3-small,1,0.5,llama-3.1-8b-instant
2,fac62bbce5555de5ca44b01442baea3f,2026-04-13 01:09:34.104908,text-embedding-3-small,1,0.5,llama-3.1-8b-instant
3,93f67870e1be18f0328798b37faf07df,2026-04-13 03:56:59.969786,text-embedding-3-small,1,0.8,openai/gpt-oss-120b
4,f45bea2857afba54057c2f1fe47ec893,2026-04-13 04:06:28.277504,text-embedding-3-small,1,0.8,llama-3.1-8b-instant
5,9c99877c5c2cde6ac3a5c05ea242b577,2026-04-13 18:12:19.101363,text-embedding-3-small,2,0.9,openai/gpt-oss-120b


## Compare 2 RAG Settings --> Provide Score Reasons

* For each record:
  * Compare 2 RAG settings --> explain why scoring is different
  * Scenario 1 without background
  * Scenario 2 with background
  * Background
    * Overall scoring distributions

#### Scenario 1: Without Background

In [ ]:
selected_hashes = ['fac62bbce5555de5ca44b01442baea3f', '93f67870e1be18f0328798b37faf07df']

In [31]:
# Build two comparison datasets for compare_2rags_async_v1
assert len(selected_hashes) == 2, "selected_hashes must contain exactly 2 config hashes"
hash1, hash2 = selected_hashes

QUERY_COL     = 'rca_records.query'
RQ_SCORE_COL  = 'rca_records.retrieval_quality_score'
AQ_SCORE_COL  = 'rca_records.answer_quality_score'
RQ_REASON_COL = 'rca_records.rq_reasoning'
AQ_REASON_COL = 'rca_records.aq_reasoning'

# --- Build per-config settings string ---
def build_settings_str(config_hash):
    r = disinct_settings[disinct_settings['config_hash'] == config_hash].iloc[0]
    return ", ".join(f"{col}: {r[col]}" for col in ['embedding_model', 'top_n_retrieval', 'semantic_weight', 'answer_gen_llm'])

rag1_settings_str = build_settings_str(hash1)
rag2_settings_str = build_settings_str(hash2)

# --- Subset and merge on query (no full-frame copy) ---
keep_cols = [QUERY_COL, RQ_SCORE_COL, AQ_SCORE_COL, RQ_REASON_COL, AQ_REASON_COL]
df_merged = (df_records_rows[df_records_rows['config_hash'] == hash1][keep_cols]
             .merge(df_records_rows[df_records_rows['config_hash'] == hash2][keep_cols],
                    on=QUERY_COL, suffixes=('_rag1', '_rag2')))

# --- Convert score cols to numeric in one call ---
score_num_cols = [RQ_SCORE_COL + s for s in ('_rag1', '_rag2')] + [AQ_SCORE_COL + s for s in ('_rag1', '_rag2')]
df_merged[score_num_cols] = df_merged[score_num_cols].apply(pd.to_numeric, errors='coerce')

# --- Add formatted columns once on merged frame, then slice ---
df_merged['rag1_settings'] = rag1_settings_str
df_merged['rag2_settings'] = rag2_settings_str
for rag, sfx in [('rag1', '_rag1'), ('rag2', '_rag2')]:
    for name, sc, rc in [('rq', RQ_SCORE_COL, RQ_REASON_COL), ('aq', AQ_SCORE_COL, AQ_REASON_COL)]:
        df_merged[f'{rag}_{name}_score_and_reasons'] = (
            '{Score: ' + df_merged[sc+sfx].astype(str) + ', Reason: ' + df_merged[rc+sfx].astype(str) + '}'
        )

rq_diff = (df_merged[RQ_SCORE_COL + '_rag1'] - df_merged[RQ_SCORE_COL + '_rag2']).abs()
aq_diff = (df_merged[AQ_SCORE_COL + '_rag1'] - df_merged[AQ_SCORE_COL + '_rag2']).abs()

final_cols = ['rag1_settings', 'rag1_rq_score_and_reasons', 'rag1_aq_score_and_reasons',
              'rag2_settings', 'rag2_rq_score_and_reasons', 'rag2_aq_score_and_reasons']
dataset_same_score = df_merged[(rq_diff == 0) & (aq_diff == 0)][final_cols].reset_index(drop=True)
dataset_diff_score = df_merged[(rq_diff != 0) | (aq_diff != 0)][final_cols].reset_index(drop=True)

print(f"dataset_same_score: {len(dataset_same_score)} records")
print(f"dataset_diff_score: {len(dataset_diff_score)} records")
display(dataset_same_score.head(3))
display(dataset_diff_score.head(3))

dataset_same_score: 13 records
dataset_diff_score: 17 records


,rag1_settings,rag1_rq_score_and_reasons,rag1_aq_score_and_reasons,rag2_settings,rag2_rq_score_and_reasons,rag2_aq_score_and_reasons
0,"embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content mirro...","{Score: 2, Reason: The AI’s answer correctly a...","embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is id...","{Score: 2, Reason: The AI’s answer addresses t..."
1,"embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is id...","{Score: 2, Reason: The AI’s answer addresses t...","embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The context directly addres...","{Score: 2, Reason: The AI’s answer correctly i..."
2,"embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is id...","{Score: 2, Reason: The AI’s answer is relevant...","embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is id...","{Score: 2, Reason: The AI's answer correctly e..."


,rag1_settings,rag1_rq_score_and_reasons,rag1_aq_score_and_reasons,rag2_settings,rag2_rq_score_and_reasons,rag2_aq_score_and_reasons
0,"embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content fully...","{Score: 2, Reason: The AI’s answer is relevant...","embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is id...","{Score: 3, Reason: The AI’s answer is fully re..."
1,"embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The user asks whether a bus...","{Score: 2, Reason: The AI’s answer is relevant...","embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content match...","{Score: 3, Reason: The AI’s answer fully addre..."
2,"embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is id...","{Score: 2, Reason: The AI’s answer correctly s...","embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is id...","{Score: 3, Reason: The AI’s answer directly ad..."


In [37]:
import importlib, utils_root_cause
importlib.reload(utils_root_cause)
from utils_root_cause import run_compare_2rags_async_v1

exp_llm = ChatGroq(
    groq_api_key=os.environ["GROQ_TOKEN"],
    model_name="openai/gpt-oss-20b",
    temperature=0.78
)

df_compare_output = await run_compare_2rags_async_v1(dataset_diff_score, exp_llm)

print(f"Processed {len(df_compare_output)} records")
display(df_compare_output.head(3))

Processed 17 records


,rag1_settings,rag1_rq_score_and_reasons,rag1_aq_score_and_reasons,rag2_settings,rag2_rq_score_and_reasons,rag2_aq_score_and_reasons,score_difference_root_causes,lower_answer_quality_root_causes
0,"embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content fully...","{Score: 2, Reason: The AI’s answer is relevant...","embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is id...","{Score: 3, Reason: The AI’s answer is fully re...",RAG2’s larger GPT‑OSS 120B LLM and higher sema...,Score‑2 answers often miss key points because ...
1,"embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The user asks whether a bus...","{Score: 2, Reason: The AI’s answer is relevant...","embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content match...","{Score: 3, Reason: The AI’s answer fully addre...",RAG2 uses a larger LLM (120B) and higher seman...,Lower scores arise when the LLM fails to captu...
2,"embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is id...","{Score: 2, Reason: The AI’s answer correctly s...","embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is id...","{Score: 3, Reason: The AI’s answer directly ad...",RAG2’s larger 120B LLM and higher semantic_wei...,RAG1’s 8B model and lower semantic focus limit...


In [38]:
folder_id = os.getenv('GOOGLE_DRIVE_FOLDER_ID')
upload_to_google_drive(df_compare_output, folder_id, 'compare_2rags_v1.xlsx')

Uploaded file ID: 1BVtPTpo5V1fFvZfR8S4zX0snbmEoyVtX
